# 추천 모델 실험

계좌 fast부터 시작한다. 섹션 순서대로 실행한다.

1. 준비
2. 사람 판정
3. 기준선 쌍대 정확도
4. LogisticRegression 학습
5. 기준선과 비교

## 1. 준비

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

RECOMMEND_DIR = Path.cwd()
if RECOMMEND_DIR.name != "recommend":
    RECOMMEND_DIR = Path.cwd() / "ai" / "recommend"
PROCESSED_DIR = RECOMMEND_DIR.parent / "data" / "processed"
sys.path.insert(0, str(RECOMMEND_DIR / "account"))
sys.path.insert(0, str(RECOMMEND_DIR))

from baseline import RATE_FULL_GAP, WEIGHTS

PAIR_DATE = "20261020"
FEATURES = list(WEIGHTS)

auto_pairs = pd.read_csv(PROCESSED_DIR / f"account_pairs_auto_{PAIR_DATE}.csv", encoding="utf-8-sig", dtype={"label": str})
judge_pairs = pd.read_csv(PROCESSED_DIR / f"account_pairs_judge_{PAIR_DATE}.csv", encoding="utf-8-sig", dtype={"label": str, "judge": str})
print(f"자동 라벨 {len(auto_pairs)}쌍, 사람 판정 후보 {len(judge_pairs)}쌍")
judge_pairs["split"].value_counts()

## 2. 사람 판정

아래 표를 보고 `LABELS`에 판정을 적는다.

- 값: `"A"` (A가 낫다), `"B"` (B가 낫다), `"같음"`
- 판정할 때 기준선 점수는 보지 않는다.
- 판정한 사람은 `JUDGE`에 적는다. 팀원이 나눠 판정하면 각자 `LABELS`를 채운 뒤 합친다.
- 판정 파일 날짜(`PAIR_DATE`)를 바꾸면 쌍 번호가 달라질 수 있다.

In [ ]:
judge_view = judge_pairs[[
    "pair_id", "purpose", "save_trm", "visit", "group",
    "a_kor_co_nm", "a_fin_prdt_nm", "a_effective_rate", "a_visit_required",
    "b_kor_co_nm", "b_fin_prdt_nm", "b_effective_rate", "b_visit_required",
]].rename(columns={
    "purpose": "목적", "save_trm": "기간", "visit": "방문", "group": "그룹",
    "a_kor_co_nm": "A 금융사", "a_fin_prdt_nm": "A 상품", "a_effective_rate": "A 수익률", "a_visit_required": "A 방문 필요",
    "b_kor_co_nm": "B 금융사", "b_fin_prdt_nm": "B 상품", "b_effective_rate": "B 수익률", "b_visit_required": "B 방문 필요",
})
pd.set_option("display.max_rows", 100)
judge_view

In [ ]:
JUDGE = ""
LABELS = {
    # "AH0001": "A",
}

judge_pairs["label"] = judge_pairs["pair_id"].map(LABELS)
judge_pairs["judge"] = np.where(judge_pairs["label"].notna(), JUDGE, "")
labeled = judge_pairs[judge_pairs["label"].isin(["A", "B"])]
print(f"판정 {judge_pairs['label'].notna().sum()} / {len(judge_pairs)}쌍 (같음 제외 {len(labeled)}쌍)")
labeled["split"].value_counts()

## 3. 기준선 쌍대 정확도

기준선은 `a_baseline`과 `b_baseline` 중 큰 쪽을 고른다. 점수가 같으면 맞힌 것으로 세지 않는다.

In [ ]:
def baseline_choice(pairs):
    return np.select(
        [pairs["a_baseline"] > pairs["b_baseline"], pairs["a_baseline"] < pairs["b_baseline"]],
        ["A", "B"],
        default="같음",
    )


def accuracy(pairs, choice):
    if len(pairs) == 0:
        return float("nan")
    return float((choice == pairs["label"].to_numpy()).mean())


print(f"자동 라벨 쌍: {accuracy(auto_pairs, baseline_choice(auto_pairs)):.3f}")
for split in ["train", "eval"]:
    part = labeled[labeled["split"] == split]
    print(f"사람 판정 {split} ({len(part)}쌍): {accuracy(part, baseline_choice(part)):.3f}")

## 4. LogisticRegression 학습

- 입력은 피처 차이 `A − B`, 정답은 A가 나으면 1이다.
- 절편은 두지 않는다. A와 B의 순서를 바꾸면 예측도 정확히 뒤집혀야 하기 때문이다.
- 학습에는 자동 라벨 쌍과 사람 판정 train 쌍을 쓴다. eval 쌍은 쓰지 않는다.

In [ ]:
def feature_diff(pairs):
    return np.column_stack([pairs[f"a_{name}"] - pairs[f"b_{name}"] for name in FEATURES])


def target(pairs):
    return (pairs["label"] == "A").astype(int).to_numpy()


train = pd.concat([auto_pairs, labeled[labeled["split"] == "train"]], ignore_index=True)
model = LogisticRegression(fit_intercept=False, C=1.0)
model.fit(feature_diff(train), target(train))

coef = pd.Series(model.coef_[0], index=FEATURES)
comparison = pd.DataFrame({"기준선": pd.Series(WEIGHTS), "학습": coef})
comparison["학습 (금리=0.5로 맞춤)"] = coef / coef["rate"] * WEIGHTS["rate"]
comparison

In [ ]:
visit_in_rate_gap = coef["visit_required"] / coef["rate"] * RATE_FULL_GAP
baseline_visit = WEIGHTS["visit_required"] / WEIGHTS["rate"] * RATE_FULL_GAP
print(f"방문 필요 1회 = 금리 {abs(visit_in_rate_gap):.2f}%p (기준선 {abs(baseline_visit):.2f}%p)")

## 5. 기준선과 비교

In [ ]:
def model_choice(pairs):
    if len(pairs) == 0:
        return np.array([])
    return np.where(model.predict(feature_diff(pairs)) == 1, "A", "B")


rows = []
for name, part in [
    ("자동 라벨", auto_pairs),
    ("사람 train", labeled[labeled["split"] == "train"]),
    ("사람 eval", labeled[labeled["split"] == "eval"]),
]:
    rows.append({
        "쌍": name,
        "개수": len(part),
        "기준선": accuracy(part, baseline_choice(part)),
        "모델": accuracy(part, model_choice(part)),
    })
pd.DataFrame(rows)

- 채택 기준: 사람 eval 쌍에서 모델 정확도가 기준선보다 높아야 한다.
- eval 쌍이 8개뿐이라 한두 개 차이는 우연일 수 있다.